# Workshop 3A: Asymmetric Cryptography & Hashes Lab (Solutions Guide)

**Topic**: Concurrency, Operating Systems, & Security — Asymmetric Encryption & Hashes  
**Target Duration**: 20–30 minutes  
**Objective**: Implement Toy RSA to explore key modular limits, execute standard library RSA key generation and encryption, and analyze the Avalanche Effect of cryptographic hash functions.

---  
## Exercise A: Toy RSA and Modulus Boundaries

RSA relies on asymmetric key pairs. In this exercise, we generate keys using small primes and observe modular boundaries.

### 1. Run the RSA script
Run the cell below to see key generation and encryption/decryption at work.

In [ ]:
p, q = 61, 53
n = p * q           # Modulus n (shared publicly)
totient = (p-1)*(q-1)

e = 17              # Public exponent
d = pow(e, -1, totient)  # Private exponent d

print(f"Public Key (e, n):  ({e}, {n})")
print(f"Private Key (d, n): ({d}, {n})")

# Message to encrypt
msg = 42

# Encryption: C = M^e mod n
ciphertext = pow(msg, e, n)

# Decryption: M = C^d mod n
decrypted = pow(ciphertext, d, n)

print(f"Original Message: {msg}")
print(f"Ciphertext:       {ciphertext}")
print(f"Decrypted:        {decrypted}")

### 2. Modulus Overflow Coding Task
**Task**: Modify the cell below by changing the message `msg` to a value larger than the modulus `n = 3233` (for example, set `msg = 4000`). Run the cell, see what value is outputted, and explain the mathematical reason why RSA has this limitation.

In [ ]:
# Solution: set msg to 4000
msg = 4000

ciphertext = pow(msg, e, n)
decrypted = pow(ciphertext, d, n)

print(f"Original Message: {msg}")
print(f"Decrypted:        {decrypted}")

### Answer & Explanation:
* **Observation**: Setting `msg = 4000` decrypts as `767`.
* **Explanation**: RSA arithmetic operates inside a finite group modulo $n$ ($\mathbb{Z}_n$). The decryption calculation restores the message as $M' = C^d \equiv M^{e \cdot d} \equiv M \pmod n$. 
  * If $M \ge n$, the equation wraps around. The decrypted output will be $M \bmod n$.
  * Here, $4000 \bmod 3233 = 767$. Therefore, RSA message blocks must always be strictly smaller than the modulus ($M < n$).

---  
## Exercise B: Real-World RSA with Libraries

In production, we use industry-standard libraries to generate large key sizes (e.g. 2048-bit or 4096-bit) and perform RSA with optimal padding (OAEP) to protect against mathematical attacks.

### 1. Complete the RSA Encryption/Decryption Script
Review the code below which generates a 2048-bit RSA key and encrypts a message. Write the missing line to decrypt the message using the private key.

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes

# 1. Generate an RSA private key (2048-bit)
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)
public_key = private_key.public_key()

# 2. Message to encrypt
message = b"Secret Session Key 123"

# 3. Encrypt using the public key and OAEP padding
ciphertext = public_key.encrypt(
    message,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)
print(f"Ciphertext Length in Bytes: {len(ciphertext)}")

# 4. Decrypt using the private key and matching OAEP padding
decrypted = private_key.decrypt(
    ciphertext,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

print(f"Decrypted: {decrypted.decode()}")
assert decrypted == message, "Asymmetric decryption failed!"

---  
## Exercise C: Cryptographic Hashes & The Avalanche Effect

Cryptographic hash functions are one-way functions that map input strings of arbitrary length to fixed-size outputs. They must be collision-resistant.

### 1. Observe the Avalanche Effect
Complete the Python script below to calculate and display the MD5 hash of two slightly different input strings using Python's standard `hashlib` library.

In [ ]:
import hashlib

input1 = "Password123"
input2 = "password123"

# Calculate hashes
hash1 = hashlib.md5(input1.encode()).hexdigest()
hash2 = hashlib.md5(input2.encode()).hexdigest()

print(f"Hash of '{input1}': {hash1}")
print(f"Hash of '{input2}': {hash2}")

### Discussion Questions & Solutions:
1. **What is the Avalanche Effect, and why is it important for the security of hash functions?**
   * **Answer**: The Avalanche Effect is the property where a tiny change in the input (like changing a single bit or character) results in a completely different and uncorrelated hash digest. This is essential for security because it ensures that attackers cannot predict changes in the hash or derive any clues about the plaintext input by comparing hashes.
2. **Why is MD5 considered broken and no longer suitable for digital signatures or file verification?**
   * **Answer**: Because of collision vulnerabilities. In 2004, researchers showed how to generate two different files that yield the exact same MD5 hash (hash collisions). This allows attackers to forge digital signatures or substitute benign files for malicious ones without altering the MD5 signature verification.